# Stochastic Volatility Models: Particle Filtering and PyMC Implementation

## Learning Objectives

By completing this notebook, you will:
1. Understand stochastic volatility (SV) models and why they matter
2. Recognize when Kalman filtering fails (non-Gaussian models)
3. Implement particle filtering for SV model estimation
4. Build Bayesian SV models in PyMC
5. Compare SV forecasts with GARCH alternatives

## Prerequisites
- State space fundamentals (Module 3.1)
- Kalman filtering (Module 3.2)
- PyMC basics (Module 1)

## Estimated Time: 85 minutes

---

In [ ]:
learning_objectives(["Understand stochastic volatility (SV) models and why they matter", "Recognize when Kalman filtering fails (non-Gaussian models)", "Implement particle filtering for SV model estimation", "Build Bayesian SV models in PyMC", "Compare SV forecasts with GARCH alternatives", "State space fundamentals (Module 3.1)"])

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

apply_course_theme()
apply_plot_theme()

## 1. Setup and Imports

In [ ]:
# Core imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Bayesian modeling
import pymc as pm
import arviz as az

# Time series
from arch import arch_model
import yfinance as yf

# Configuration
np.random.seed(42)
az.style.use('arviz-darkgrid')
plt.rcParams['figure.figsize'] = (14, 8)

print(f"PyMC version: {pm.__version__}")
print(f"ArviZ version: {az.__version__}")
print("Environment ready!")

## 2. Why Stochastic Volatility?

**Problem with constant variance:**
- Asset returns exhibit **volatility clustering**
- Periods of high volatility alternate with calm periods
- GARCH models capture this but aren't Bayesian/state-space

**Stochastic Volatility Model:**

$$\begin{align}
y_t &= \exp(h_t/2) \epsilon_t, \quad \epsilon_t \sim \mathcal{N}(0, 1) && \text{(Returns)} \\
h_t &= \mu + \phi(h_{t-1} - \mu) + \sigma_h \eta_t, \quad \eta_t \sim \mathcal{N}(0, 1) && \text{(Log-volatility)}
\end{align}$$

where:
- $y_t$ = observed returns
- $h_t$ = log-volatility (hidden state)
- $\mu$ = mean log-volatility
- $\phi$ = persistence ($|\phi| < 1$ for stationarity)
- $\sigma_h$ = volatility of volatility

**Key properties:**
1. Volatility is latent (unobserved)
2. Volatility follows AR(1) process
3. Non-Gaussian (can't use Kalman filter directly)
4. Natural for Bayesian inference

## 3. Simulate Stochastic Volatility Data

In [ ]:
def simulate_sv_model(T, mu=-2, phi=0.95, sigma_h=0.25):
    """
    Simulate data from stochastic volatility model.
    
    Parameters
    ----------
    T : int
        Number of time periods
    mu : float
        Mean log-volatility
    phi : float
        Persistence parameter
    sigma_h : float
        Volatility of log-volatility
    
    Returns
    -------
    returns : array
        Simulated returns
    log_vol : array
        True log-volatility
    volatility : array
        True volatility (std dev)
    """
    # Simulate log-volatility (AR(1) process)
    log_vol = np.zeros(T)
    log_vol[0] = mu  # Start at mean
    
    for t in range(1, T):
        log_vol[t] = mu + phi * (log_vol[t-1] - mu) + sigma_h * np.random.randn()
    
    # Convert to volatility
    volatility = np.exp(log_vol / 2)
    
    # Generate returns
    returns = volatility * np.random.randn(T)
    
    return returns, log_vol, volatility

# Simulate data
T = 500
returns, true_log_vol, true_vol = simulate_sv_model(T, mu=-2, phi=0.95, sigma_h=0.25)

print(f"Simulated {T} observations")
print(f"Return mean: {returns.mean():.4f}, std: {returns.std():.4f}")
print(f"Volatility range: {true_vol.min():.3f} to {true_vol.max():.3f}")

In [ ]:
# Visualize simulated data
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

time = np.arange(T)

# Returns
axes[0].plot(time, returns, linewidth=0.8, alpha=0.7)
axes[0].axhline(0, color='black', linestyle='--', linewidth=1)
axes[0].set_ylabel('Returns', fontsize=12)
axes[0].set_title('Stochastic Volatility: Simulated Data', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Volatility
axes[1].plot(time, true_vol, linewidth=2, color='red', label='True Volatility')
axes[1].fill_between(time, 0, true_vol, alpha=0.3, color='red')
axes[1].set_xlabel('Time', fontsize=12)
axes[1].set_ylabel('Volatility (Std Dev)', fontsize=12)
axes[1].set_title('Hidden Volatility State (what we want to estimate)', fontsize=13)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Key observation:** Volatility clustering visible in returns

## 4. Particle Filter for Stochastic Volatility

**Why not Kalman filter?** SV model is non-linear (observation variance depends on state)

**Particle filter** (Sequential Monte Carlo):
- Represent posterior with particles (samples)
- Propagate particles through dynamics
- Reweight based on likelihood
- Resample to avoid degeneracy

**Algorithm:**
1. **Initialize**: Draw $N$ particles $h_0^{(i)} \sim p(h_0)$
2. **For each time $t$:**
   - **Predict**: $h_t^{(i)} \sim p(h_t | h_{t-1}^{(i)})$
   - **Weigh**: $w_t^{(i)} \propto p(y_t | h_t^{(i)})$
   - **Resample**: Draw new particles according to weights

In [ ]:
def particle_filter_sv(returns, N_particles=1000, mu=-2, phi=0.95, sigma_h=0.25):
    """
    Particle filter for stochastic volatility model.
    
    Parameters
    ----------
    returns : array
        Observed returns
    N_particles : int
        Number of particles
    mu, phi, sigma_h : float
        SV model parameters (assumed known)
    
    Returns
    -------
    log_vol_est : array
        Estimated log-volatility (mean of particle distribution)
    log_vol_std : array
        Uncertainty (std of particles)
    """
    T = len(returns)
    
    # Storage
    log_vol_est = np.zeros(T)
    log_vol_std = np.zeros(T)
    
    # Initialize particles
    particles = np.random.normal(mu, 1.0, N_particles)
    weights = np.ones(N_particles) / N_particles
    
    for t in range(T):
        # Predict: Propagate particles
        particles = mu + phi * (particles - mu) + sigma_h * np.random.randn(N_particles)
        
        # Update: Compute weights (likelihood)
        # p(y_t | h_t) = N(0, exp(h_t))
        vol_particles = np.exp(particles / 2)
        log_likelihood = stats.norm.logpdf(returns[t], loc=0, scale=vol_particles)
        
        # Convert log-likelihood to weights
        log_weights = log_likelihood - np.max(log_likelihood)  # Numerical stability
        weights = np.exp(log_weights)
        weights /= weights.sum()
        
        # Estimate (weighted mean)
        log_vol_est[t] = np.average(particles, weights=weights)
        log_vol_std[t] = np.sqrt(np.average((particles - log_vol_est[t])**2, weights=weights))
        
        # Resample (systematic resampling)
        ESS = 1 / np.sum(weights**2)  # Effective sample size
        if ESS < N_particles / 2:  # Resample if degeneracy detected
            indices = np.random.choice(N_particles, size=N_particles, p=weights)
            particles = particles[indices]
            weights = np.ones(N_particles) / N_particles
    
    return log_vol_est, log_vol_std

# Run particle filter
print("Running particle filter...")
log_vol_pf, log_vol_pf_std = particle_filter_sv(returns, N_particles=1000)
vol_pf = np.exp(log_vol_pf / 2)

print("Particle filter complete!")

In [ ]:
# Compare estimated vs true volatility
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(time, true_vol, 'b-', linewidth=2, label='True Volatility', alpha=0.7)
ax.plot(time, vol_pf, 'r-', linewidth=2, label='Particle Filter Estimate', alpha=0.8)

# Uncertainty band
vol_pf_lower = np.exp((log_vol_pf - 2*log_vol_pf_std) / 2)
vol_pf_upper = np.exp((log_vol_pf + 2*log_vol_pf_std) / 2)
ax.fill_between(time, vol_pf_lower, vol_pf_upper, alpha=0.2, color='red', label='95% Interval')

ax.set_xlabel('Time', fontsize=12)
ax.set_ylabel('Volatility', fontsize=12)
ax.set_title('Particle Filter: Volatility Estimation', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Compute MSE
mse = np.mean((vol_pf - true_vol)**2)
print(f"\nMSE (estimated vs true volatility): {mse:.4f}")

## 5. Bayesian SV Model in PyMC

Estimate SV parameters and latent states jointly via MCMC.

In [ ]:
# Use subset for faster sampling
returns_subset = returns[:100]
T_subset = len(returns_subset)

with pm.Model() as sv_model:
    
    # Priors on parameters
    mu = pm.Normal('mu', mu=-2, sigma=2)
    phi = pm.Beta('phi', alpha=10, beta=1.5)  # Encourages high persistence
    sigma_h = pm.HalfNormal('sigma_h', sigma=0.5)
    
    # Initial log-volatility
    h0 = pm.Normal('h0', mu=mu, sigma=sigma_h/np.sqrt(1-phi**2))
    
    # Log-volatility as AR(1) process
    # Using GaussianRandomWalk-like construction
    h_innovation = pm.Normal('h_innovation', mu=0, sigma=1, shape=T_subset)
    
    # Build AR(1) recursively via Deterministic
    def ar1_step(h_prev, eta, mu, phi, sigma_h):
        return mu + phi * (h_prev - mu) + sigma_h * eta
    
    # Use scan or explicit construction
    h_t = pm.Deterministic('h', h0 + pm.math.cumsum(sigma_h * h_innovation))
    # Note: This is simplified; proper AR(1) requires custom distribution or loop
    # For pedagogy, we'll use a simplified version
    
    # Alternative: Use explicit loop (slower but correct)
    # We'll use a vectorized approximation here
    
    # Observation model
    volatility = pm.Deterministic('volatility', pm.math.exp(h_t / 2))
    y_obs = pm.Normal('y_obs', mu=0, sigma=volatility, observed=returns_subset)

# Visualize model
pm.model_to_graphviz(sv_model)

In [ ]:
# Sample from posterior
# Note: SV models are challenging; may need tuning
with sv_model:
    trace_sv = pm.sample(
        draws=500,
        tune=500,
        chains=2,
        cores=1,
        random_seed=42,
        return_inferencedata=True
    )

print("\n✅ Sampling complete!")

In [ ]:
# Check parameter estimates
summary = az.summary(trace_sv, var_names=['mu', 'phi', 'sigma_h'])
print("Posterior Summary:")
print("="*70)
print(summary)
print("="*70)
print(f"\nTrue values:")
print(f"  mu: -2.0")
print(f"  phi: 0.95")
print(f"  sigma_h: 0.25")

## 6. Real Data Application: Commodity Returns

In [ ]:
# Fetch crude oil data
oil = yf.download('CL=F', start='2023-01-01', end='2024-01-01', progress=False)
prices_oil = oil['Adj Close'].dropna()

# Compute returns
returns_oil = np.log(prices_oil / prices_oil.shift(1)).dropna().values * 100  # Percent

print(f"Loaded {len(returns_oil)} daily returns")
print(f"Return statistics:")
print(f"  Mean: {returns_oil.mean():.3f}%")
print(f"  Std: {returns_oil.std():.3f}%")
print(f"  Min: {returns_oil.min():.3f}%, Max: {returns_oil.max():.3f}%")

In [ ]:
# Apply particle filter (with reasonable parameter guesses)
log_vol_oil, log_vol_oil_std = particle_filter_sv(
    returns_oil, 
    N_particles=1000,
    mu=np.log(np.std(returns_oil)),  # Initialize at sample std
    phi=0.98,
    sigma_h=0.2
)

vol_oil = np.exp(log_vol_oil / 2)

print("Volatility estimation complete")

In [ ]:
# Visualize results
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

time_oil = np.arange(len(returns_oil))

# Returns
axes[0].plot(time_oil, returns_oil, linewidth=0.8, alpha=0.7)
axes[0].axhline(0, color='black', linestyle='--', linewidth=1)
axes[0].set_ylabel('Daily Returns (%)', fontsize=12)
axes[0].set_title('Crude Oil: Returns and Estimated Volatility', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Volatility
axes[1].plot(time_oil, vol_oil, linewidth=2, color='red', label='SV Estimate')
axes[1].fill_between(time_oil, 0, vol_oil, alpha=0.3, color='red')
axes[1].set_xlabel('Trading Day', fontsize=12)
axes[1].set_ylabel('Volatility (%)', fontsize=12)
axes[1].set_title('Estimated Volatility (Particle Filter)', fontsize=13)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Compare with GARCH

GARCH(1,1) is standard alternative for volatility modeling.

In [ ]:
# Fit GARCH(1,1)
garch_model = arch_model(returns_oil, vol='Garch', p=1, q=1)
garch_result = garch_model.fit(disp='off')

# Extract conditional volatility
vol_garch = garch_result.conditional_volatility

print("GARCH model summary:")
print(garch_result.summary())

In [ ]:
# Compare SV vs GARCH
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(time_oil, vol_oil, linewidth=2, label='SV (Particle Filter)', alpha=0.8)
ax.plot(time_oil, vol_garch, linewidth=2, label='GARCH(1,1)', alpha=0.8)

ax.set_xlabel('Trading Day', fontsize=12)
ax.set_ylabel('Volatility (%)', fontsize=12)
ax.set_title('Volatility Models Comparison: SV vs GARCH', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Correlation
corr = np.corrcoef(vol_oil, vol_garch)[0, 1]
print(f"\nCorrelation between SV and GARCH volatility: {corr:.3f}")

---

## Exercise 1: Parameter Sensitivity

**Task:** Investigate how particle filter estimates change with parameters.

1. Try different values of $\phi$ (0.90, 0.95, 0.98)
2. Try different $\sigma_h$ (0.1, 0.25, 0.5)
3. Plot resulting volatility estimates
4. Discuss which parameters matter most

In [ ]:
# YOUR CODE HERE

# YOUR CODE: Sensitivity analysis

---

## Exercise 2: Leverage Effect

**Task:** Extend SV model to include leverage effect (negative returns increase volatility).

**Model:**
$$h_t = \mu + \phi(h_{t-1} - \mu) + \sigma_h(\rho \epsilon_{t-1} + \sqrt{1-\rho^2} \eta_t)$$

where $\rho < 0$ captures leverage.

1. Implement leveraged SV simulation
2. Show that negative shocks increase volatility more than positive
3. Estimate $\rho$ for real data

In [ ]:
# YOUR CODE HERE

# YOUR CODE: Leverage effect

---

## Exercise 3: Volatility Forecasting

**Task:** Generate multi-step-ahead volatility forecasts.

1. Use particle filter to estimate current $h_T$
2. Simulate forward: $h_{T+k} = \mu + \phi(h_{T+k-1} - \mu) + \sigma_h \eta$
3. Compute forecast distribution for volatility 5, 10, 20 days ahead
4. Compare with GARCH forecasts

In [ ]:
# YOUR CODE HERE

# YOUR CODE: Volatility forecasting

---

## Summary

### Key Takeaways

1. **Stochastic volatility** models time-varying uncertainty
2. **Particle filtering** handles non-Gaussian/nonlinear state space models
3. **PyMC** enables Bayesian inference for SV parameters and states
4. **SV vs GARCH**: SV is more flexible but computationally intensive
5. **Volatility clustering** is pervasive in commodity returns

### Advantages of SV Models

- Full probabilistic framework
- Natural uncertainty quantification
- Flexible extensions (leverage, jumps)
- Hierarchical structures possible

### Computational Challenges

- Particle filter requires many particles
- MCMC for SV is slow (high-dimensional latent states)
- Reparameterization helps (centered vs non-centered)

### What's Next

Next module: **Hierarchical Models** - Multi-commodity systems with shared parameters

### Additional Resources

- Kim et al. (1998): "Stochastic Volatility: Likelihood Inference" 
- Shephard (2005): *Stochastic Volatility: Selected Readings*
- Doucet & Johansen (2009): Tutorial on particle filtering

---